# Experiment 25: Synthetic Identity Deep Dive

This experiment extends the winning Experiment 23B identity-encoding approach.

Candidates:
- **25A**: Identity encoding + digit/number-structure features
- **25B**: Identity encoding + selected higher-order identity features
- **25C**: Identity encoding + digit/number structure + higher-order identities

Reference points:
- Experiment 23B local ROC-AUC: **0.945243**
- Submission 08 Kaggle ROC-AUC: **0.94754**

The goal is to find additional synthetic structure that can improve the current leaderboard benchmark.


In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "DataCompetition":
    PROJECT_ROOT = Path.home() / "Documents" / "DataCompetition"

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train.csv"

if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Training data not found: {TRAIN_PATH}")

train = pd.read_csv(TRAIN_PATH)

TARGET = "Will_Buy_EV"
ID_COL = "id"

NUM_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

CAT_COLS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

X_raw = train.drop(columns=[TARGET, ID_COL])
target_values = train[TARGET].astype(str).str.strip()

target_map = {
    "No": 0,
    "Yes": 1,
}

unknown_targets = sorted(set(target_values.unique()) - set(target_map))

if unknown_targets:
    raise ValueError(
        f"Unexpected target values found: {unknown_targets}"
    )

y = target_values.map(target_map).astype(np.int8)

BASE_X = pd.get_dummies(
    X_raw,
    columns=CAT_COLS,
    dtype=np.int8
)

print(f"Train shape: {train.shape}")
print(f"Base feature count: {BASE_X.shape[1]}")
print(f"Target mean: {y.mean():.6f}")


Train shape: (668665, 15)
Base feature count: 24
Target mean: 0.174645


In [2]:

RANDOM_STATE = 42
N_SPLITS = 3
SMOOTHING = 20.0

XGB_PARAMS = dict(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.04,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


def fit_oof_xgb(X, y):
    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    oof = np.zeros(len(X), dtype=np.float64)

    for fold, (tr_idx, va_idx) in enumerate(
        skf.split(X, y),
        start=1
    ):
        print(f"Training fold {fold}/{N_SPLITS}...")

        model = XGBClassifier(**XGB_PARAMS)

        model.fit(
            X.iloc[tr_idx],
            y.iloc[tr_idx],
        )

        oof[va_idx] = model.predict_proba(
            X.iloc[va_idx]
        )[:, 1]

        print(f"Fold {fold}/{N_SPLITS} complete")

    return oof


def add_identity_features_oof(
    base_X,
    raw_X,
    y,
    identity_cols,
    smoothing=20.0,
):
    out = base_X.copy()

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    global_mean = float(y.mean())

    for col in identity_cols:
        values = raw_X[col]

        te = np.zeros(len(raw_X), dtype=np.float64)
        freq = np.zeros(len(raw_X), dtype=np.float64)

        for tr_idx, va_idx in skf.split(raw_X, y):
            tr_values = values.iloc[tr_idx]
            va_values = values.iloc[va_idx]
            tr_y = y.iloc[tr_idx]

            stats = pd.DataFrame({
                "key": tr_values.to_numpy(),
                "target": tr_y.to_numpy(),
            })

            grouped = (
                stats
                .groupby("key", dropna=False)["target"]
                .agg(["sum", "count"])
            )

            sums = (
                va_values
                .map(grouped["sum"])
                .fillna(0.0)
                .to_numpy()
            )

            counts = (
                va_values
                .map(grouped["count"])
                .fillna(0.0)
                .to_numpy()
            )

            te[va_idx] = (
                sums + smoothing * global_mean
            ) / (
                counts + smoothing
            )

            freq[va_idx] = counts / len(tr_idx)

        out[f"{col}__identity_te"] = te
        out[f"{col}__identity_freq"] = freq

    return out


def add_pair_identity_features_oof(
    base_X,
    raw_X,
    y,
    pairs,
    smoothing=20.0,
):
    out = base_X.copy()

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    global_mean = float(y.mean())

    for cols in pairs:
        name = "__".join(cols)

        keys = (
            raw_X[list(cols)]
            .astype(str)
            .agg("||".join, axis=1)
        )

        te = np.zeros(len(raw_X), dtype=np.float64)
        freq = np.zeros(len(raw_X), dtype=np.float64)

        for tr_idx, va_idx in skf.split(raw_X, y):
            tr_keys = keys.iloc[tr_idx]
            va_keys = keys.iloc[va_idx]
            tr_y = y.iloc[tr_idx]

            stats = pd.DataFrame({
                "key": tr_keys.to_numpy(),
                "target": tr_y.to_numpy(),
            })

            grouped = (
                stats
                .groupby("key", dropna=False)["target"]
                .agg(["sum", "count"])
            )

            sums = (
                va_keys
                .map(grouped["sum"])
                .fillna(0.0)
                .to_numpy()
            )

            counts = (
                va_keys
                .map(grouped["count"])
                .fillna(0.0)
                .to_numpy()
            )

            te[va_idx] = (
                sums + smoothing * global_mean
            ) / (
                counts + smoothing
            )

            freq[va_idx] = counts / len(tr_idx)

        out[f"{name}__pair_te"] = te
        out[f"{name}__pair_freq"] = freq

    return out


def add_digit_features(base_X, raw_X):
    out = base_X.copy()

    for col in NUM_COLS:
        s = pd.to_numeric(
            raw_X[col],
            errors="coerce"
        )

        filled = s.fillna(-999999999.0)
        rounded = np.round(filled).astype(np.int64)
        absolute = np.abs(filled)

        out[f"{col}__is_integer"] = (
            np.isclose(
                filled,
                np.round(filled)
            ).astype(np.int8)
        )

        out[f"{col}__mod2"] = (
            np.mod(rounded, 2)
            .astype(np.int8)
        )

        out[f"{col}__mod5"] = (
            np.mod(rounded, 5)
            .astype(np.int8)
        )

        out[f"{col}__mod10"] = (
            np.mod(rounded, 10)
            .astype(np.int8)
        )

        out[f"{col}__last_digit"] = (
            np.mod(
                np.floor(absolute).astype(np.int64),
                10
            )
            .astype(np.int8)
        )

        out[f"{col}__last2"] = (
            np.mod(
                np.floor(absolute).astype(np.int64),
                100
            )
            .astype(np.int16)
        )

        safe_values = absolute.where(
            absolute > 0,
            1.0
        )

        out[f"{col}__digit_count"] = (
            np.floor(
                np.log10(safe_values)
            )
            .add(1)
            .clip(lower=1, upper=12)
            .astype(np.int8)
        )

    return out


In [3]:

print("")
print("=" * 70)
print("BUILDING EXPERIMENT 23B REFERENCE FEATURE VIEW")
print("=" * 70)

X_identity = add_identity_features_oof(
    BASE_X,
    X_raw,
    y,
    NUM_COLS,
    smoothing=SMOOTHING,
)

print(f"23B-style feature count: {X_identity.shape[1]}")



BUILDING EXPERIMENT 23B REFERENCE FEATURE VIEW
23B-style feature count: 38


In [4]:

print("")
print("=" * 70)
print("25A: IDENTITY + DIGIT / NUMBER STRUCTURE")
print("=" * 70)

X_25A = add_digit_features(
    X_identity,
    X_raw,
)

print(f"25A feature count: {X_25A.shape[1]}")

oof_25A = fit_oof_xgb(
    X_25A,
    y,
)

score_25A = roc_auc_score(
    y,
    oof_25A,
)

print(f"25A OOF ROC-AUC: {score_25A:.6f}")
print(f"Delta vs 23B: {score_25A - 0.945243:+.6f}")



25A: IDENTITY + DIGIT / NUMBER STRUCTURE
25A feature count: 87
Training fold 1/3...
Fold 1/3 complete
Training fold 2/3...
Fold 2/3 complete
Training fold 3/3...
Fold 3/3 complete
25A OOF ROC-AUC: 0.944766
Delta vs 23B: -0.000477


In [5]:

PAIR_COLS = [
    ("Age", "Annual_Income_USD"),
    ("Annual_Income_USD", "Daily_Commute_km"),
    ("Daily_Commute_km", "Charging_Stations_Near_Work"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
    ("Number_of_Cars_Owned", "Annual_Income_USD"),
    ("Age", "Daily_Commute_km"),
    ("Age", "Charging_Stations_Near_Home"),
    ("Age", "Charging_Stations_Near_Work"),
    ("Annual_Income_USD", "Charging_Stations_Near_Home"),
    ("Annual_Income_USD", "Charging_Stations_Near_Work"),
]

print("")
print("=" * 70)
print("25B: IDENTITY + HIGHER-ORDER IDENTITY FEATURES")
print("=" * 70)

X_25B = add_pair_identity_features_oof(
    X_identity,
    X_raw,
    y,
    PAIR_COLS,
    smoothing=SMOOTHING,
)

print(f"25B feature count: {X_25B.shape[1]}")

oof_25B = fit_oof_xgb(
    X_25B,
    y,
)

score_25B = roc_auc_score(
    y,
    oof_25B,
)

print(f"25B OOF ROC-AUC: {score_25B:.6f}")
print(f"Delta vs 23B: {score_25B - 0.945243:+.6f}")



25B: IDENTITY + HIGHER-ORDER IDENTITY FEATURES
25B feature count: 58
Training fold 1/3...
Fold 1/3 complete
Training fold 2/3...
Fold 2/3 complete
Training fold 3/3...
Fold 3/3 complete
25B OOF ROC-AUC: 0.944937
Delta vs 23B: -0.000306


In [6]:

print("")
print("=" * 70)
print("25C: IDENTITY + DIGIT + HIGHER-ORDER IDENTITY")
print("=" * 70)

X_25C = add_digit_features(
    X_25B,
    X_raw,
)

print(f"25C feature count: {X_25C.shape[1]}")

oof_25C = fit_oof_xgb(
    X_25C,
    y,
)

score_25C = roc_auc_score(
    y,
    oof_25C,
)

print(f"25C OOF ROC-AUC: {score_25C:.6f}")
print(f"Delta vs 23B: {score_25C - 0.945243:+.6f}")



25C: IDENTITY + DIGIT + HIGHER-ORDER IDENTITY
25C feature count: 107
Training fold 1/3...
Fold 1/3 complete
Training fold 2/3...
Fold 2/3 complete
Training fold 3/3...
Fold 3/3 complete
25C OOF ROC-AUC: 0.945031
Delta vs 23B: -0.000212


In [7]:

results = pd.DataFrame({
    "experiment": [
        "23B_reference",
        "25A",
        "25B",
        "25C",
    ],
    "features": [
        X_identity.shape[1],
        X_25A.shape[1],
        X_25B.shape[1],
        X_25C.shape[1],
    ],
    "oof_roc_auc": [
        0.945243,
        score_25A,
        score_25B,
        score_25C,
    ],
})

results["delta_vs_23B"] = (
    results["oof_roc_auc"] - 0.945243
)

results = (
    results
    .sort_values(
        "oof_roc_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

print("")
print("=" * 70)
print("EXPERIMENT 25 RESULTS")
print("=" * 70)
print(results.to_string(index=False))
print("=" * 70)

winner = results.iloc[0]

print("")
print(
    f"Winner: {winner['experiment']} "
    f"| OOF ROC-AUC: {winner['oof_roc_auc']:.6f} "
    f"| Delta: {winner['delta_vs_23B']:+.6f}"
)

print("")
print("Kaggle benchmark to beat: 0.947540")
print("No submission is created by this notebook.")



EXPERIMENT 25 RESULTS
   experiment  features  oof_roc_auc  delta_vs_23B
23B_reference        38     0.945243      0.000000
          25C       107     0.945031     -0.000212
          25B        58     0.944937     -0.000306
          25A        87     0.944766     -0.000477

Winner: 23B_reference | OOF ROC-AUC: 0.945243 | Delta: +0.000000

Kaggle benchmark to beat: 0.947540
No submission is created by this notebook.
